# Silver Layer: Districts

This notebook reads the `bronze.districts` table, applies data type corrections (casting `id` to `IntegerType`, `ingestion_date` to `DateType`, `ingestion_timestamp` to `TimestampType`), trims string columns, removes records with null IDs, and writes the cleaned data to the `silver.districts` table.

In [0]:
from pyspark.sql.functions import trim, col, to_date, current_timestamp
from pyspark.sql.types import IntegerType, StringType, TimestampType, DateType
from datetime import datetime

In [0]:
# Define bronze and silver table paths
bronze_table = "trips_carris_metropolitana.bronze.districts"
silver_table = "trips_carris_metropolitana.silver.districts"

# Read bronze data
df = spark.table(bronze_table)

# Cast columns to appropriate Silver types
df = (
    df
    .withColumn("id", col("id").cast(IntegerType()))
    .withColumn("name", col("name").cast(StringType()))
    .withColumn("ingestion_date", to_date(col("ingestion_date"), "yyyy-MM-dd"))
    .withColumn("ingestion_timestamp", col("ingestion_timestamp").cast(TimestampType()))
)

# Trim string columns
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Remove records with missing district ID (if any)
df = df.filter(col("id").isNotNull())

# Write to silver table (overwrite)
df.write.mode("overwrite").format("delta").saveAsTable(silver_table)

In [0]:
%sql
SELECT * FROM trips_carris_metropolitana.silver.districts LIMIT 5;